# MA-EZV2 Demo

This notebook demonstrates instantiation of the MA-EZV2 policy, running MCTS, and plotting simple diagnostics. It is a non-executed, ready-to-run notebook.


In [ ]:
import yaml
import os
import torch
import matplotlib.pyplot as plt

from paperAssignments.Assignments1_50.CA10.integration.lightzero_adapter import (
    LightZeroAdapter,
)

cfg_path = os.path.join(
    "..",
    "paperAssignments",
    "Assignments1-50",
    "CA10",
    "configs",
    "ma_ezv2_default.yaml",
)
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

adapter = LightZeroAdapter(cfg, device="cpu")
obs = torch.randn(1, cfg["model"]["obs_dim"])
info = adapter.infer(obs)
print("root value:", info["value"])

In [ ]:
res = adapter.search(obs, sims=30, topk=6)
visits = res["visits"].squeeze(0).numpy()
policy = res["policy"].squeeze(0).numpy()

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(visits)), visits)
plt.title("Visit counts")
plt.subplot(1, 2, 2)
plt.bar(range(len(policy)), policy)
plt.title("Policy from visits")
plt.tight_layout()
plt.savefig("../pictures/ma_ezv2_demo.png", dpi=200)
print("Saved figure to ../pictures/ma_ezv2_demo.png")

## Training step example

Below is a template cell showing how to call adapter.training_step in a training loop. It's not executed here.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch

ds = TensorDataset(
    torch.randn(32, cfg["model"]["obs_dim"]),
    torch.randn(32, cfg["model"]["joint_action_dim"]),
    torch.softmax(torch.randn(32, cfg["model"]["joint_action_dim"]), dim=-1),
    torch.randn(32),
    torch.randn(32),
    torch.randn(32),
)
loader = DataLoader(ds, batch_size=8)
optim = torch.optim.Adam(adapter.policy.parameters(), lr=1e-3)
loss_weights = cfg.get("loss_weights", {})
for batch in loader:
    obs_b, actions_b, pi_b, v_b, r_b, z_b = [b for b in batch]
    batch_dict = {
        "obs": obs_b,
        "actions": actions_b,
        "pi_target": pi_b,
        "v_target": v_b,
        "r_target": r_b,
        "z_target": z_b,
    }
    loss_val, loss_terms = adapter.training_step(batch_dict, loss_weights, optim)
    print("loss", loss_val)